#  C3-Pipeline: PySpark ETL Pipeline

## Overview
This notebook performs the Extract-Transform-Load pipeline:
1. **Load** raw CSV data into Spark DataFrames
2. **Clean** data (timestamp parsing, dedup, referential integrity)
3. **Engineer features** (response lag, turnaround times, friction scores)
4. **Aggregate** into analytical tables (`agg_consult_friction`, `agg_encounter_summary`)
5. **Output** to both Parquet (portfolio) and CSV (Tableau)

## Setup

In [8]:

# !pip install pyspark
# PySpark, Engine to process Big Data

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, FloatType
)
from pyspark.sql.window import Window
import os

# Initialize Spark session
# Note: Set timezone to UTC to avoid daylight saving issues
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('C3-Pipeline-ETL') \
    .config('spark.sql.legacy.timeParserPolicy', 'LEGACY') \
    .config('spark.sql.session.timeZone', 'UTC') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN') # show only warnings and errors logs

# Paths
RAW_DIR = os.path.join('..', 'data', 'raw')
PARQUET_DIR = os.path.join('..', 'data', 'processed', 'parquet')
CSV_DIR = os.path.join('..', 'data', 'processed', 'csv')

os.makedirs(PARQUET_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)


print('✅ Spark session initialized.')
print(f'   Spark version: {spark.version}')

✅ Spark session initialized.
   Spark version: 4.0.3


---
## Step 1: Load Raw Data

In [9]:

# Load all four raw CSV files into Spark DataFrames
df_encounters = spark.read.csv(
    os.path.join(RAW_DIR, 'dim_encounters.csv'),
    header=True, inferSchema=True # guess data types
)

df_consults = spark.read.csv(
    os.path.join(RAW_DIR, 'fact_consult_orders.csv'),
    header=True, inferSchema=True
)

df_comms = spark.read.csv(
    os.path.join(RAW_DIR, 'fact_communication_logs.csv'),
    header=True, inferSchema=True
)

df_completions = spark.read.csv(
    os.path.join(RAW_DIR, 'fact_consult_completions.csv'),
    header=True, inferSchema=True
)

print('📂 Raw data loaded:')
print(f'   dim_encounters:           {df_encounters.count():>10,} rows')
print(f'   fact_consult_orders:      {df_consults.count():>10,} rows')
print(f'   fact_communication_logs:  {df_comms.count():>10,} rows')
print(f'   fact_consult_completions: {df_completions.count():>10,} rows')

📂 Raw data loaded:
   dim_encounters:               10,000 rows
   fact_consult_orders:          29,824 rows
   fact_communication_logs:      82,404 rows
   fact_consult_completions:     25,362 rows


In [10]:
# Inspect schemas
print('Schema: dim_encounters')
df_encounters.printSchema()
print('\nSchema: fact_consult_orders')
df_consults.printSchema()
print('\nSchema: fact_communication_logs')
df_comms.printSchema()
print('\nSchema: fact_consult_completions')
df_completions.printSchema()

Schema: dim_encounters
root
 |-- encounter_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- patient_age: integer (nullable = true)
 |-- patient_sex: string (nullable = true)
 |-- admit_timestamp: string (nullable = true)
 |-- discharge_timestamp: string (nullable = true)
 |-- discharge_disposition: string (nullable = true)
 |-- primary_diagnosis_code: string (nullable = true)
 |-- primary_diagnosis_desc: string (nullable = true)
 |-- admitting_unit: string (nullable = true)
 |-- attending_provider_id: string (nullable = true)


Schema: fact_consult_orders
root
 |-- consult_order_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- requesting_provider_id: string (nullable = true)
 |-- requesting_provider_specialty: string (nullable = true)
 |-- target_specialty: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- order_status: string (nullable = true)


Schema: fact_commu

---
## Step 2: Data Cleaning

### 2.1 Timestamp Parsing
Cast all timestamp columns from strings to proper `TimestampType`.

In [11]:
# dim_encounters: cast timestamps
df_encounters = df_encounters \
    .withColumn('admit_timestamp', F.to_timestamp('admit_timestamp', 'yyyy-MM-dd HH:mm:ss')) \
    .withColumn('discharge_timestamp', F.to_timestamp('discharge_timestamp', 'yyyy-MM-dd HH:mm:ss'))

# fact_consult_orders: cast timestamps
df_consults = df_consults \
    .withColumn('order_timestamp', F.to_timestamp('order_timestamp', 'yyyy-MM-dd HH:mm:ss'))

# fact_communication_logs: cast timestamps
df_comms = df_comms \
    .withColumn('message_sent_timestamp', F.to_timestamp('message_sent_timestamp', 'yyyy-MM-dd HH:mm:ss')) \
    .withColumn('message_read_timestamp', F.to_timestamp('message_read_timestamp', 'yyyy-MM-dd HH:mm:ss'))

# fact_consult_completions: cast timestamps
df_completions = df_completions \
    .withColumn('bedside_arrival_timestamp', F.to_timestamp('bedside_arrival_timestamp', 'yyyy-MM-dd HH:mm:ss')) \
    .withColumn('note_signed_timestamp', F.to_timestamp('note_signed_timestamp', 'yyyy-MM-dd HH:mm:ss'))

print('✅ All timestamp columns parsed.')

✅ All timestamp columns parsed.


### 2.2 Deduplication
Check for and remove any duplicate primary key values.

In [12]:
# Check for duplicates on primary keys
tables_pks = {
    'dim_encounters': ('encounter_id', df_encounters),
    'fact_consult_orders': ('consult_order_id', df_consults),
    'fact_communication_logs': ('message_id', df_comms),
    'fact_consult_completions': ('completion_id', df_completions),
}

for table_name, (pk, df) in tables_pks.items():
    total = df.count()
    distinct = df.select(pk).distinct().count()
    dupes = total - distinct
    print(f'{table_name}: {total:,} rows, {distinct:,} distinct {pk}, {dupes} duplicates')
    if dupes > 0:
        print(f'   ⚠️ Removing {dupes} duplicate rows...')
        # Keep first occurrence
        w = Window.partitionBy(pk).orderBy(F.monotonically_increasing_id())
        df = df.withColumn('_row_num', F.row_number().over(w)).filter(F.col('_row_num') == 1).drop('_row_num')

# Re-assign (in case duplicates were removed)
df_encounters = tables_pks['dim_encounters'][1]
df_consults = tables_pks['fact_consult_orders'][1]
df_comms = tables_pks['fact_communication_logs'][1]
df_completions = tables_pks['fact_consult_completions'][1]

print('\n✅ Deduplication complete.')

dim_encounters: 10,000 rows, 10,000 distinct encounter_id, 0 duplicates
fact_consult_orders: 29,824 rows, 29,824 distinct consult_order_id, 0 duplicates
fact_communication_logs: 82,404 rows, 82,404 distinct message_id, 0 duplicates
fact_consult_completions: 25,362 rows, 25,362 distinct completion_id, 0 duplicates

✅ Deduplication complete.


### 2.3 Referential Integrity Validation
Verify that every foreign key maps to a valid primary key.

In [13]:
# Check: every encounter_id in fact_consult_orders exists in dim_encounters
enc_ids = df_encounters.select('encounter_id').distinct()
consult_enc_ids = df_consults.select('encounter_id').distinct()
orphan_consults = consult_enc_ids.subtract(enc_ids)
print(f'Orphan encounter_ids in fact_consult_orders: {orphan_consults.count()}')

# Check: every consult_order_id in fact_communication_logs exists in fact_consult_orders
consult_ids = df_consults.select('consult_order_id').distinct()
comm_consult_ids = df_comms.select('consult_order_id').distinct()
orphan_comms = comm_consult_ids.subtract(consult_ids)
print(f'Orphan consult_order_ids in fact_communication_logs: {orphan_comms.count()}')

# Check: every consult_order_id in fact_consult_completions exists in fact_consult_orders
comp_consult_ids = df_completions.select('consult_order_id').distinct()
orphan_comps = comp_consult_ids.subtract(consult_ids)
print(f'Orphan consult_order_ids in fact_consult_completions: {orphan_comps.count()}')

print('\n✅ Referential integrity validated.')

Orphan encounter_ids in fact_consult_orders: 0
Orphan consult_order_ids in fact_communication_logs: 0
Orphan consult_order_ids in fact_consult_completions: 0

✅ Referential integrity validated.


---
## Step 3: Feature Engineering

### 3.1 Communication Log Metrics

In [ ]:
# response_lag_minutes: time between message sent and message read
# NULL if the message was never read
df_comms = df_comms.withColumn( # Adds one column
    'response_lag_minutes',
    # Spark's equivalent of an IF-THEN-ELSE statement
    F.when(
        F.col('message_read_timestamp').isNotNull(),
        F.round(
            (F.unix_timestamp('message_read_timestamp') - F.unix_timestamp('message_sent_timestamp')) / 60.0,
            2
        )
    ).otherwise(None)
)

print('✅ response_lag_minutes calculated.')
print(f'   Non-null response lags: {df_comms.filter(F.col("response_lag_minutes").isNotNull()).count():,}')
print(f'   Null (unread) messages: {df_comms.filter(F.col("response_lag_minutes").isNull()).count():,}')

# Show sample statistics
df_comms.filter(F.col('response_lag_minutes').isNotNull()) \
    .groupBy('channel') \
    .agg(
        F.round(F.avg('response_lag_minutes'), 1).alias('avg_lag_min'),
        F.round(F.stddev('response_lag_minutes'), 1).alias('std_lag_min'),
    ) \
    .orderBy('avg_lag_min') \
    .show()

✅ response_lag_minutes calculated.
   Non-null response lags: 73,046
   Null (unread) messages: 9,358
+--------------------+-----------+-----------+
|             channel|avg_lag_min|std_lag_min|
+--------------------+-----------+-----------+
|   Vocera Badge Call|        3.1|        1.8|
|     Secure App Chat|        8.2|        4.7|
|        Legacy Pager|       25.3|       14.3|
|Phone Call to Office|       40.3|       19.8|
+--------------------+-----------+-----------+



### 3.2 Consult Completion Metrics

In [15]:
# Join completions with consult orders to get order_timestamp
df_completions_enriched = df_completions.join(
    df_consults.select('consult_order_id', 'order_timestamp', 'target_specialty', 'priority'),
    on='consult_order_id',
    how='inner'
)

# time_to_bedside_hours: order to bedside arrival
df_completions_enriched = df_completions_enriched.withColumn(
    'time_to_bedside_hours',
    F.round(
        (F.unix_timestamp('bedside_arrival_timestamp') - F.unix_timestamp('order_timestamp')) / 3600.0,
        2
    )
)

# time_to_note_signed_hours: order to note signed
df_completions_enriched = df_completions_enriched.withColumn(
    'time_to_note_signed_hours',
    F.round(
        (F.unix_timestamp('note_signed_timestamp') - F.unix_timestamp('order_timestamp')) / 3600.0,
        2
    )
)

# note_writing_duration_minutes: bedside to note signed
df_completions_enriched = df_completions_enriched.withColumn(
    'note_writing_duration_minutes',
    F.round(
        (F.unix_timestamp('note_signed_timestamp') - F.unix_timestamp('bedside_arrival_timestamp')) / 60.0,
        2
    )
)

print('✅ Consult completion metrics calculated.')
print('\nAverage turnaround by specialty:')
df_completions_enriched \
    .groupBy('target_specialty') \
    .agg(
        F.count('*').alias('count'),
        F.round(F.avg('time_to_bedside_hours'), 2).alias('avg_hours_to_bedside'),
        F.round(F.avg('time_to_note_signed_hours'), 2).alias('avg_hours_to_note_signed'),
    ) \
    .orderBy(F.desc('avg_hours_to_bedside')) \
    .show(12)

✅ Consult completion metrics calculated.

Average turnaround by specialty:
+------------------+-----+--------------------+------------------------+
|  target_specialty|count|avg_hours_to_bedside|avg_hours_to_note_signed|
+------------------+-----+--------------------+------------------------+
|        Psychiatry| 2585|                7.94|                    8.72|
|        Cardiology| 2463|                7.47|                    8.25|
|        Nephrology| 2589|                 6.9|                    7.67|
|   Palliative Care| 2537|                 6.1|                    6.88|
|         Neurology| 2551|                5.96|                    6.75|
|       Pulmonology| 2532|                 5.5|                    6.28|
|          Oncology| 2554|                5.34|                     6.1|
|Infectious Disease| 2566|                4.88|                    5.66|
|       Orthopedics| 2501|                4.58|                    5.35|
|  Gastroenterology| 2484|                4.34|  

### 3.3 Aggregated Table: `agg_consult_friction`

One row per consult order with messaging friction metrics.

In [18]:
# Find the dominant (most frequent) channel per consult
channel_counts = df_comms.groupBy('consult_order_id', 'channel') \
    .agg(F.count('*').alias('channel_count'))

w_channel = Window.partitionBy('consult_order_id').orderBy(F.desc('channel_count'))
dominant_channel = channel_counts \
    .withColumn('rn', F.row_number().over(w_channel)) \
    .filter(F.col('rn') == 1) \
    .select('consult_order_id', F.col('channel').alias('dominant_channel'))

# Main aggregation
agg_friction = df_comms.groupBy('consult_order_id').agg(
    F.count('*').alias('total_messages_sent'),
    F.sum(F.when(F.col('message_read_timestamp').isNotNull(), 1).otherwise(0)).alias('total_messages_read'),
    F.sum(F.when(F.col('message_read_timestamp').isNull(), 1).otherwise(0)).alias('unread_message_count'),
    F.min('message_sent_timestamp').alias('first_message_timestamp'),
    F.max('message_sent_timestamp').alias('last_message_timestamp'),
    F.round(F.avg('response_lag_minutes'), 2).alias('avg_response_lag_minutes'),
)

# Join with dominant channel
agg_friction = agg_friction.join(dominant_channel, on='consult_order_id', how='left')

# Add friction_score (alias for total_messages_sent)
agg_friction = agg_friction.withColumn('friction_score', F.col('total_messages_sent'))

print(f'✅ agg_consult_friction created: {agg_friction.count():,} rows')
print('\nFriction score distribution( total messages sent to get a response):')
agg_friction.groupBy('friction_score').count().orderBy('friction_score').show(10)

✅ agg_consult_friction created: 29,824 rows

Friction score distribution( total messages sent to get a response):
+--------------+-----+
|friction_score|count|
+--------------+-----+
|             1|10422|
|             2| 6817|
|             3| 4382|
|             4| 2902|
|             5| 1884|
|             6| 1182|
|             7|  798|
|             8| 1437|
+--------------+-----+



### 3.4 Aggregated Table: `agg_encounter_summary`

One row per encounter with consult and communication summary metrics.

In [19]:
# Total consults ordered per encounter
consults_per_enc = df_consults.groupBy('encounter_id').agg(
    F.count('*').alias('total_consults_ordered'),
    F.sum(F.when(F.col('order_status') == 'Completed', 1).otherwise(0)).alias('total_consults_completed'),
)

# Total messages per encounter (join through consult_order_id)
msgs_per_consult = df_comms.groupBy('consult_order_id').agg(
    F.count('*').alias('msg_count')
)
consult_msgs = df_consults.select('consult_order_id', 'encounter_id') \
    .join(msgs_per_consult, on='consult_order_id', how='left')
msgs_per_enc = consult_msgs.groupBy('encounter_id').agg(
    F.sum('msg_count').alias('total_nurse_messages_sent')
)

# Average time to bedside per encounter
bedside_per_consult = df_completions_enriched.select(
    'consult_order_id', 'time_to_bedside_hours'
)
consult_bedside = df_consults.select('consult_order_id', 'encounter_id') \
    .join(bedside_per_consult, on='consult_order_id', how='left')
bedside_per_enc = consult_bedside.groupBy('encounter_id').agg(
    F.round(F.avg('time_to_bedside_hours'), 2).alias('avg_time_to_bedside_hours'),
    F.round(F.sum('time_to_bedside_hours'), 2).alias('total_time_to_bedside_hours'),
)

# Length of stay
los = df_encounters.select(
    'encounter_id',
    F.round(
        (F.unix_timestamp('discharge_timestamp') - F.unix_timestamp('admit_timestamp')) / 86400.0,
        2
    ).alias('length_of_stay_days')
)

# Combine all into agg_encounter_summary
agg_encounter_summary = consults_per_enc \
    .join(msgs_per_enc, on='encounter_id', how='left') \
    .join(bedside_per_enc, on='encounter_id', how='left') \
    .join(los, on='encounter_id', how='left')

# Estimated excess bed days: total time waiting for specialists / 24
agg_encounter_summary = agg_encounter_summary.withColumn(
    'estimated_excess_bed_days',
    F.round(F.col('total_time_to_bedside_hours') / 24.0, 2)
)

# Fill nulls for encounters with no messages or completions
agg_encounter_summary = agg_encounter_summary \
    .fillna({'total_nurse_messages_sent': 0, 'avg_time_to_bedside_hours': 0,
             'total_time_to_bedside_hours': 0, 'estimated_excess_bed_days': 0})

print(f'✅ agg_encounter_summary created: {agg_encounter_summary.count():,} rows')
agg_encounter_summary.describe().show()

✅ agg_encounter_summary created: 6,000 rows
+-------+------------+----------------------+------------------------+-------------------------+-------------------------+---------------------------+-------------------+-------------------------+
|summary|encounter_id|total_consults_ordered|total_consults_completed|total_nurse_messages_sent|avg_time_to_bedside_hours|total_time_to_bedside_hours|length_of_stay_days|estimated_excess_bed_days|
+-------+------------+----------------------+------------------------+-------------------------+-------------------------+---------------------------+-------------------+-------------------------+
|  count|        6000|                  6000|                    6000|                     6000|                     6000|                       6000|               5976|                     6000|
|   mean|        NULL|     4.970666666666666|                   4.227|                   13.734|        5.872591666666689|         24.963666666666647|  3.97691265060241

---
## Step 4: Save Processed Data

Output all cleaned tables and aggregated tables in both Parquet and CSV formats.

In [21]:
# Define all tables to save
tables_to_save = {
    'dim_encounters': df_encounters,
    'fact_consult_orders': df_consults,
    'fact_communication_logs': df_comms,
    'fact_consult_completions': df_completions_enriched,
    'agg_consult_friction': agg_friction,
    'agg_encounter_summary': agg_encounter_summary,
}

for table_name, df in tables_to_save.items():
    # Save as Parquet(Binary compressed format)
    parquet_path = os.path.join(PARQUET_DIR, table_name)
    df.write.parquet(parquet_path, mode='overwrite')
    print(f'✅ {table_name} → Parquet saved ({df.count():,} rows)')
    
    # Save as CSV (for Tableau Public upload)
    csv_path = os.path.join(CSV_DIR, table_name)
    df.coalesce(1).write.csv(csv_path, header=True, mode='overwrite')
    print(f'✅ {table_name} → CSV saved')

print('\n All processed data saved successfully!')

✅ dim_encounters → Parquet saved (10,000 rows)
✅ dim_encounters → CSV saved
✅ fact_consult_orders → Parquet saved (29,824 rows)
✅ fact_consult_orders → CSV saved
✅ fact_communication_logs → Parquet saved (82,404 rows)
✅ fact_communication_logs → CSV saved
✅ fact_consult_completions → Parquet saved (25,362 rows)
✅ fact_consult_completions → CSV saved
✅ agg_consult_friction → Parquet saved (29,824 rows)
✅ agg_consult_friction → CSV saved
✅ agg_encounter_summary → Parquet saved (6,000 rows)
✅ agg_encounter_summary → CSV saved

 All processed data saved successfully!


In [23]:
# Stop Spark session
spark.stop()
print('✅ Spark session stopped.')

✅ Spark session stopped.
